In [1]:
import pandas as pd

# Load both sheets
transactions = pd.read_excel(
    "../data/raw/restaurant_sales_datanew.csv.xlsx",
    sheet_name="transactions"
)

customers = pd.read_excel(
    "../data/raw/restaurant_sales_datanew.csv.xlsx",
    sheet_name="customers"
)

# First look
print("Transactions shape:", transactions.shape)
print("Customers shape:", customers.shape)
print("\nTransactions columns:")
print(transactions.dtypes)

Transactions shape: (15251, 11)
Customers shape: (100, 3)

Transactions columns:
order_id                     str
customer_id                  str
category                     str
item_name                    str
item_price                 int64
quantity                   int64
total_amt                  int64
order_date        datetime64[us]
payment_method               str
ordered_by                   str
time_of_sale                 str
dtype: object


In [2]:
print("=== Missing Values: Transactions ===")
print(transactions.isnull().sum())

print("\n=== Missing Values: Customers ===")
print(customers.isnull().sum())

=== Missing Values: Transactions ===
order_id          0
customer_id       0
category          0
item_name         0
item_price        0
quantity          0
total_amt         0
order_date        0
payment_method    0
ordered_by        0
time_of_sale      0
dtype: int64

=== Missing Values: Customers ===
customer_id    0
city           0
age_group      0
dtype: int64


In [5]:
print("Duplicate rows in transactions:", transactions.duplicated().sum())
print("Duplicate order_ids:", transactions['order_id'].duplicated().sum())
print("Duplicate rows in customers:", customers.duplicated().sum())

Duplicate rows in transactions: 0
Duplicate order_ids: 0
Duplicate rows in customers: 0


In [6]:
# Check current type
print("order_date dtype:", transactions['order_date'].dtype)

# Convert to datetime (already done, but confirm)
transactions['order_date'] = pd.to_datetime(transactions['order_date'])

# Verify date range
print("Earliest date:", transactions['order_date'].min())
print("Latest date:", transactions['order_date'].max())

order_date dtype: datetime64[us]
Earliest date: 2022-01-01 00:00:00
Latest date: 2023-12-31 00:00:00


In [7]:
# Check if total_amt = item_price × quantity
transactions['calculated_total'] = transactions['item_price'] * transactions['quantity']
mismatches = transactions[transactions['total_amt'] != transactions['calculated_total']]
print(f"Mismatches in total_amt: {len(mismatches)}")

# Drop the helper column
transactions.drop('calculated_total', axis=1, inplace=True)

Mismatches in total_amt: 0


In [8]:
print("=== item_price range ===")
print(transactions['item_price'].describe())

print("\n=== total_amt range ===")
print(transactions['total_amt'].describe())

print("\n=== quantity range ===")
print(transactions['quantity'].describe())

# Flag suspicious orders (quantity > 10 is unusual for restaurant)
high_qty = transactions[transactions['quantity'] > 9]
print(f"\nOrders with quantity > 9: {len(high_qty)}")

=== item_price range ===
count    15251.000000
mean        65.971084
std         48.410721
min         10.000000
25%         30.000000
50%         50.000000
75%         70.000000
max        200.000000
Name: item_price, dtype: float64

=== total_amt range ===
count    15251.000000
mean       199.638057
std        187.358445
min         10.000000
25%         80.000000
50%        150.000000
75%        250.000000
max       1000.000000
Name: total_amt, dtype: float64

=== quantity range ===
count    15251.000000
mean         3.020195
std          1.413745
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          5.000000
Name: quantity, dtype: float64

Orders with quantity > 9: 0


In [9]:
# Merge transactions with customer info
df = transactions.merge(customers, on='customer_id', how='left')

print("Merged shape:", df.shape)
print("Columns after merge:", df.columns.tolist())

# Check for unmatched customers
print("Null cities after merge:", df['city'].isnull().sum())

Merged shape: (15251, 13)
Columns after merge: ['order_id', 'customer_id', 'category', 'item_name', 'item_price', 'quantity', 'total_amt', 'order_date', 'payment_method', 'ordered_by', 'time_of_sale', 'city', 'age_group']
Null cities after merge: 0


In [10]:
df.to_csv('../data/cleaned/restaurant_cleaned.csv', index=False)
print("Cleaned data saved!")

Cleaned data saved!
